# Word Embeddings: From Dense Vectors to Semantic Similarity

This notebook provides a hands-on exploration of **word embeddings** — dense vector representations of words learned from data. We implement a Bag-of-Embeddings model **from scratch using NumPy** and compare it with **sklearn's TF-IDF baseline** on the IMDB sentiment classification task.

## Prerequisites
- Basic tokenization and text preprocessing
- Softmax classification and cross-entropy loss
- NumPy array operations
- sklearn basics (TfidfVectorizer, LogisticRegression)

## Dataset
[IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)

**Credits:** Andrew L. Maas, Raymond E. Daly, Peter T. Pham, Dan Huang, Andrew Y. Ng, and Christopher Potts. (2011). *Learning Word Vectors for Sentiment Analysis.* ACL.

In [ ]:
import numpy as np              # Numerical operations
import pandas as pd             # Data loading
import matplotlib.pyplot as plt # Plotting
import seaborn as sns           # Visualizations
import re                       # Text cleaning
from collections import Counter # Vocabulary building
from pathlib import Path        # File handling
import warnings                 # Suppress warnings
import kagglehub                # Download Kaggle datasets
warnings.filterwarnings('ignore')

# sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

np.random.seed(42)              # Reproducibility

## Part 1: Theory Recap — Word Embeddings

- **Distributional hypothesis:** "You shall know a word by the company it keeps" — words in similar contexts have similar meanings.
- **Dense vectors:** Unlike one-hot encoding (sparse, high-dim), embeddings map each word to a dense, low-dimensional vector of real numbers learned from data.
- **Embedding matrix:** A lookup table of shape [vocab_size x embed_dim]; each word's row is its learned vector; training updates these rows via gradient descent.
- **Similarity via dot product:** The dot product (or cosine similarity) between two embedding vectors measures semantic relatedness; geometrically close words are semantically similar.
- **Bag-of-embeddings:** Averaging all word vectors in a document produces a document vector that captures overall semantic content; this is a simple but effective baseline.

In [ ]:
# Download the IMDB 50K dataset via kagglehub (requires internet on first run)
print("Downloading IMDB dataset via kagglehub...")
path = Path(kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"))
csv_path = [f for f in path.glob("*.csv")][0]
df = pd.read_csv(csv_path, encoding='utf-8')

# Quick data inspection
print(f"\nDataset shape: {df.shape}\n")
print("--- First 5 rows ---")
print(df.head())
print("\n--- Dataset info ---")
df.info()
print("\n--- Target distribution ---")
print(df['sentiment'].value_counts())

# The dataset has 50,000 movie reviews with two columns:
#   review:     Full text of the movie review
#   sentiment:  Binary label — 'positive' or 'negative'

In [ ]:
# ------ Text Cleaning ------
def clean_text(text, max_words=200):
    """Lowercase, strip HTML, keep only letters, tokenize, truncate."""
    text = text.lower()
    text = re.sub(r'<br\\s*/?>', ' ', text)   # remove <br> tags
    text = re.sub(r'[^a-z\\s]', '', text)       # keep only letters and spaces
    tokens = text.split()
    return tokens[:max_words] if len(tokens) > max_words else tokens

# ------ Vocabulary Building ------
def build_vocab(tokenized_texts, max_vocab=5000):
    """Build word-to-id mapping from tokenized texts, keeping top-k words."""
    counter = Counter()
    for tokens in tokenized_texts:
        counter.update(tokens)
    # Reserve index 0 for <PAD> and last for <UNK>
    most_common = counter.most_common(max_vocab - 2)
    word2idx = {word: idx+1 for idx, (word, _) in enumerate(most_common)}
    word2idx['<PAD>'] = 0
    word2idx['<UNK>'] = len(word2idx)
    return word2idx

def encode(tokens, word2idx):
    """Convert token list to integer IDs; OOV words become <UNK>."""
    unk_id = word2idx['<UNK>']
    return [word2idx.get(w, unk_id) for w in tokens]

# ------ Apply preprocessing ------
print("Cleaning text...")
all_tokens = [clean_text(text) for text in df['review']]

print("Building vocabulary...")
vocab = build_vocab(all_tokens, max_vocab=5000)
vocab_size = len(vocab)
idx2word = {v: k for k, v in vocab.items()}
print(f"Vocabulary size: {vocab_size}")

print("Encoding reviews...")
encoded = [encode(tokens, vocab) for tokens in all_tokens]

# Convert labels: 'positive' -> 1, 'negative' -> 0
label_map = {'positive': 1, 'negative': 0}
y = np.array([label_map[s] for s in df['sentiment']])

# ------ Train / Test Split ------
# Store the split indices so we can align raw text for sklearn
indices = np.arange(len(df))
train_idx, test_idx, y_train, y_test = train_test_split(
    indices, y, test_size=0.2, random_state=42, stratify=y
)
X_train = [encoded[i] for i in train_idx]
X_test  = [encoded[i] for i in test_idx]

# Also keep raw text for sklearn (use original strings, not cleaned)
X_train_text = df.iloc[train_idx]['review'].values
X_test_text  = df.iloc[test_idx]['review'].values

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
print(f"Train positive ratio: {y_train.mean():.3f} | Test positive ratio: {y_test.mean():.3f}")

## Part 2: From-Scratch Implementation — Word Embedding Classifier

We build a **WordEmbeddingClassifier** using only NumPy. The model learns an embedding matrix where each word maps to a dense vector. For each review, word vectors are averaged into a document vector, then classified via softmax. This is the "bag-of-embeddings" model — the simplest form of neural word embedding.

**How it works:**
1. Look up the embedding vector for each word in the review from the embedding matrix.
2. Average all word vectors into a single document vector (mean pooling).
3. Pass the document vector through a linear layer with softmax to predict sentiment.

The embedding matrix and classification weights are trained end-to-end with **mini-batch SGD** and manual backpropagation.

In [ ]:
class WordEmbeddingClassifier:
    """
    From-scratch word embedding model using only NumPy.

    Learns a dense embedding matrix via gradient descent.
    Each review's word vectors are averaged and classified.
    """
    def __init__(self, vocab_size, embed_dim=50, lr=0.01):
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.lr = lr

        # Embedding matrix: [vocab_size, embed_dim]
        # INTERVIEW NOTE: Small random init prevents dead neurons from large initial logits
        self.E = np.random.randn(vocab_size, embed_dim) * 0.01

        # Classification layer: embed_dim -> 2 classes
        self.W = np.random.randn(2, embed_dim) * 0.01
        self.b = np.zeros(2)

    def _softmax(self, x):
        """Numerically stable softmax."""
        x_shift = x - x.max(axis=-1, keepdims=True)
        exp_x = np.exp(x_shift)
        return exp_x / exp_x.sum(axis=-1, keepdims=True)

    def _forward(self, seq):
        """Forward pass: embedding lookup -> mean pool -> classify."""
        # INTERVIEW NOTE: Mean pooling assumes all words contribute equally;
        # this loses word order but works surprisingly well.
        emb = self.E[seq].mean(axis=0)               # (embed_dim,)
        logits = self.W @ emb + self.b                # (2,)
        probs = self._softmax(logits)
        cache = {'seq': seq, 'emb': emb, 'logits': logits, 'probs': probs}
        return probs, cache

    def _backward(self, cache, y_true):
        """Manual backprop through softmax + linear + average pool."""
        seq, emb = cache['seq'], cache['emb']
        logits, probs = cache['logits'], cache['probs']

        # dL/dlogits = probs - y_onehot (derivative of CE + softmax)
        # INTERVIEW NOTE: This simplification is a common interview trick —
        # the softmax gradient cancels with the CE gradient.
        y_onehot = np.zeros(2)
        y_onehot[y_true] = 1.0
        dlogits = probs - y_onehot

        dW = np.outer(dlogits, emb)                  # (2, embed_dim)
        db = dlogits.copy()
        demb = self.W.T @ dlogits                    # (embed_dim,)

        # Embedding gradients: scatter demb / n back to each word
        dE = np.zeros_like(self.E)
        n = len(seq)
        for wid in seq:
            dE[wid] += demb / n

        return {'W': dW, 'b': db, 'E': dE}

    def _sgd_step(self, grads):
        """Vanilla SGD update."""
        self.W -= self.lr * grads['W']
        self.b -= self.lr * grads['b']
        self.E -= self.lr * grads['E']

    def fit(self, X, y, epochs=10, batch_size=32, verbose=True):
        """Train with mini-batch SGD. Returns loss per epoch."""
        n = len(X)
        losses = []
        for epoch in range(epochs):
            idxs = np.random.permutation(n)
            epoch_loss = 0.0
            n_batches = 0
            for j in range(0, n, batch_size):
                batch_idxs = idxs[j:j + batch_size]
                # Accumulate gradients over batch
                param_names = ['W', 'b', 'E']
                grad_acc = {k: np.zeros_like(getattr(self, k)) for k in param_names}
                batch_loss = 0.0

                for bi in batch_idxs:
                    probs, cache = self._forward(X[bi])
                    loss = -np.log(probs[y[bi]] + 1e-15)
                    batch_loss += loss
                    grads = self._backward(cache, y[bi])
                    for k in grad_acc:
                        grad_acc[k] += grads[k]

                m = len(batch_idxs)
                for k in grad_acc:
                    grad_acc[k] /= m
                self._sgd_step(grad_acc)
                epoch_loss += batch_loss / m
                n_batches += 1

            avg_loss = epoch_loss / n_batches
            losses.append(avg_loss)
            if verbose:
                print(f"  Epoch {epoch+1}/{epochs} -- Avg Loss: {avg_loss:.4f}")
        return losses

    def predict(self, X):
        return np.array([np.argmax(self._forward(seq)[0]) for seq in X])

    def predict_proba(self, X):
        return np.array([self._forward(seq)[0] for seq in X])

    def get_vector(self, word_id):
        """Return embedding vector for a given word index."""
        return self.E[word_id]

    def most_similar(self, word_id, idx2word, topn=10):
        """Find most similar words by cosine similarity."""
        vec = self.E[word_id]
        norms = np.linalg.norm(self.E, axis=1)
        sims = (self.E @ vec) / (norms * np.linalg.norm(vec) + 1e-15)
        top_indices = np.argsort(sims)[-topn-1:][::-1]
        return [(idx2word[i], sims[i]) for i in top_indices if i != word_id]

In [ ]:
# Use a subset of training data for faster from-scratch training
subset_size = 2000
np.random.seed(42)
subset_idxs = np.random.choice(len(X_train), subset_size, replace=False)
X_train_sub = [X_train[i] for i in subset_idxs]
y_train_sub = y_train[subset_idxs]

print(f"Training from-scratch model on {len(X_train_sub)} reviews...\n")

# Initialize model with embedding dimension 100
scratch_model = WordEmbeddingClassifier(
    vocab_size=vocab_size, embed_dim=100, lr=0.01
)

# Train
losses = scratch_model.fit(X_train_sub, y_train_sub, epochs=10, batch_size=32)

# Evaluate on the full test set
print("\nEvaluating on test set (10,000 reviews)...")
y_pred_scratch = scratch_model.predict(X_test)

acc_s  = accuracy_score(y_test, y_pred_scratch)
prec_s = precision_score(y_test, y_pred_scratch)
rec_s  = recall_score(y_test, y_pred_scratch)
f1_s   = f1_score(y_test, y_pred_scratch)

print("\n" + "=" * 52)
print("  FROM-SCRATCH WORD EMBEDDING RESULTS")
print("=" * 52)
print(f"  Accuracy : {acc_s:.4f}")
print(f"  Precision: {prec_s:.4f}")
print(f"  Recall   : {rec_s:.4f}")
print(f"  F1 Score : {f1_s:.4f}")
print("=" * 52)

# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(losses) + 1), losses, marker='o', linestyle='-', color='#2c3e50')
plt.title("Training Loss -- From-Scratch Word Embeddings", fontsize=14, fontweight='bold')
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.xticks(range(1, len(losses) + 1))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ---- Show learned embedding similarities ----
print("\nMost similar words (learned embeddings):")
for word in ["good", "bad", "movie", "film", "great", "terrible"]:
    if word in vocab:
        wid = vocab[word]
        sims = scratch_model.most_similar(wid, idx2word, topn=5)
        sim_str = ", ".join([f"{w} ({s:.3f})" for w, s in sims])
        print(f"  '{word}': {sim_str}")
    else:
        print(f"  '{word}': <not in vocabulary>")

## Part 3: Sklearn Implementation — TF-IDF + Logistic Regression

While word embeddings learn dense, low-dimensional representations, sklearn's `TfidfVectorizer` creates sparse bag-of-ngram features. Unlike dense word embeddings, TF-IDF is high-dimensional and sparse, with each dimension corresponding to a specific n-gram. Combined with `LogisticRegression`, this is a strong text classification baseline.

**Key differences from embeddings:**
- TF-IDF features are sparse and interpretable (each dimension = a word/phrase).
- TF-IDF uses frequency weighting (term frequency * inverse document frequency).
- TF-IDF does not capture semantic similarity between different words (e.g., "great" and "excellent" are orthogonal).

In [ ]:
# --- Sklearn baseline: Tfidf + LogisticRegression ---

# Use raw original text (not cleaned) -- TfidfVectorizer handles tokenization
print("Vectorizing with Tfidf (n-gram range 1-2)...")
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=20000,
                             stop_words='english', sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf  = vectorizer.transform(X_test_text)
print(f"  Train TF-IDF matrix: {X_train_tfidf.shape}")
print(f"  Test  TF-IDF matrix: {X_test_tfidf.shape}")

# Train logistic regression
print("\nTraining LogisticRegression...")
lr_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred_lr = lr_model.predict(X_test_tfidf)

acc_lr  = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr  = recall_score(y_test, y_pred_lr)
f1_lr   = f1_score(y_test, y_pred_lr)

print("\n" + "=" * 52)
print("  SKLEARN TF-IDF + LOGISTIC REGRESSION RESULTS")
print("=" * 52)
print(f"  Accuracy : {acc_lr:.4f}")
print(f"  Precision: {prec_lr:.4f}")
print(f"  Recall   : {rec_lr:.4f}")
print(f"  F1 Score : {f1_lr:.4f}")
print("=" * 52)

# ---- Direct Comparison ----
print("\n" + "=" * 52)
print("  MODEL COMPARISON")
print("=" * 52)
print(f"  {'Metric':<15} {'Scratch (Emb)':<20} {'Sklearn (Tfidf+LR)':<20}")
print(f"  {'-'*15} {'-'*20} {'-'*20}")
print(f"  {'Accuracy':<15} {acc_s:<20.4f} {acc_lr:<20.4f}")
print(f"  {'Precision':<15} {prec_s:<20.4f} {prec_lr:<20.4f}")
print(f"  {'Recall':<15} {rec_s:<20.4f} {rec_lr:<20.4f}")
print(f"  {'F1 Score':<15} {f1_s:<20.4f} {f1_lr:<20.4f}")
print("=" * 52)

# Classification report for sklearn (more detail)
print("\nSklearn -- Classification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['negative', 'positive']))

In [ ]:
# ---- Visualization 1: Confusion Matrix Comparison ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

cm_scratch = confusion_matrix(y_test, y_pred_scratch)
cm_sklearn = confusion_matrix(y_test, y_pred_lr)

sns.heatmap(cm_scratch, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative','Positive'],
            yticklabels=['Negative','Positive'], ax=axes[0])
axes[0].set_title('Scratch -- Confusion Matrix', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_sklearn, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Negative','Positive'],
            yticklabels=['Negative','Positive'], ax=axes[1])
axes[1].set_title('Sklearn -- Confusion Matrix', fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

# ---- Visualization 2: Bar Chart Side-by-Side Comparison ----
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
scratch_scores = [acc_s, prec_s, rec_s, f1_s]
sklearn_scores = [acc_lr, prec_lr, rec_lr, f1_lr]

x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, scratch_scores, width, label='Scratch (Word Emb)', color='#2c3e50')
bars2 = ax.bar(x + width/2, sklearn_scores, width, label='Sklearn (Tfidf + LR)', color='#27ae60')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Scratch vs Sklearn -- Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend(loc='lower right')
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## Part 4: Hyperparameter Experiments

The two most influential hyperparameters for our word embedding model are:

1. **Embedding dimension (`embed_dim`)** — Controls the capacity of the representation. Too small and the model cannot capture enough signal; too large and it may overfit or train slowly.
2. **Batch size (`batch_size`)** — Determines how many samples are processed before each parameter update. Smaller batches provide noisier gradients but faster convergence; larger batches give smoother gradients but may converge to sharper minima.

We vary both and measure test accuracy with 3-fold cross-validation (on a subset) to find the optimal configuration.

In [ ]:
print("Hyperparameter sweep -- this may take a minute...\n")

# Use a smaller CV subset for speed
cv_size = 800
np.random.seed(42)
cv_idxs = np.random.choice(len(X_train), cv_size, replace=False)
X_cv = [X_train[i] for i in cv_idxs]
y_cv = y_train[cv_idxs]

from sklearn.model_selection import StratifiedKFold

def cv_accuracy(model_class, X, y, **kwargs):
    """Evaluate model with 3-fold stratified CV, return mean accuracy."""
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    accs = []
    for train_i, val_i in skf.split(X, y):
        X_tr = [X[i] for i in train_i]
        y_tr = y[train_i]
        X_va = [X[i] for i in val_i]
        y_va = y[val_i]
        m = model_class(**kwargs)
        m.fit(X_tr, y_tr, epochs=5, batch_size=16, verbose=False)
        preds = m.predict(X_va)
        accs.append(accuracy_score(y_va, preds))
    return np.mean(accs), np.std(accs)

# Experiment 1: Vary embedding dimension (fixed batch_size=32)
embed_dims = [20, 50, 100]
print("Varying embedding dimension (batch_size=32)...")
dim_means, dim_stds = [], []
for dim in embed_dims:
    mean, std = cv_accuracy(WordEmbeddingClassifier, X_cv, y_cv,
                            vocab_size=vocab_size, embed_dim=dim, lr=0.01)
    dim_means.append(mean)
    dim_stds.append(std)
    print(f"  embed_dim={dim:3d}  ->  CV Accuracy: {mean:.4f} +/- {std:.4f}")

# Experiment 2: Vary batch size (fixed embed_dim=50)
batch_sizes = [8, 32, 64]
print("\nVarying batch size (embed_dim=50)...")
bs_means, bs_stds = [], []
for bs in batch_sizes:
    mean, std = cv_accuracy(WordEmbeddingClassifier, X_cv, y_cv,
                            vocab_size=vocab_size, embed_dim=50, lr=0.01)
    bs_means.append(mean)
    bs_stds.append(std)
    print(f"  batch_size={bs:2d}  ->  CV Accuracy: {mean:.4f} +/- {std:.4f}")

# ---- Plot results ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Embedding dimension
axes[0].errorbar(embed_dims, dim_means, yerr=dim_stds, marker='o', capsize=6,
                 color='#2c3e50', linewidth=2, markersize=8)
axes[0].set_xlabel('Embedding Dimension', fontsize=11)
axes[0].set_ylabel('CV Accuracy', fontsize=11)
axes[0].set_title('Effect of Embedding Dimension on Accuracy', fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0.5, 1.0)
for dim, m, s in zip(embed_dims, dim_means, dim_stds):
    axes[0].annotate(f'{m:.3f}', (dim, m), textcoords='offset points',
                     xytext=(0, 12), ha='center', fontsize=9)

# Batch size
axes[1].errorbar(batch_sizes, bs_means, yerr=bs_stds, marker='s', capsize=6,
                 color='#8e44ad', linewidth=2, markersize=8)
axes[1].set_xlabel('Batch Size', fontsize=11)
axes[1].set_ylabel('CV Accuracy', fontsize=11)
axes[1].set_title('Effect of Batch Size on Accuracy', fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].set_xticks(batch_sizes)
axes[1].set_ylim(0.5, 1.0)
for bs, m, s in zip(batch_sizes, bs_means, bs_stds):
    axes[1].annotate(f'{m:.3f}', (bs, m), textcoords='offset points',
                     xytext=(0, 12), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## Part 5: Interview Corner

### Q: What is the difference between word embeddings and bag-of-words representations? When would you use each?

**Short answer:** Word embeddings are dense, low-dimensional vectors that capture semantic similarity between words (e.g., "great" and "excellent" are close in embedding space). Bag-of-words (BOW) is sparse, high-dimensional, and treats each word as an independent feature with no notion of similarity between different words.

**Narrative explanation:**

Consider the sentences: *"The film was great"* and *"The movie was excellent".*

With **bag-of-words**, these sentences share zero features (assuming no overlap between "great"/"excellent" and "film"/"movie") — they look completely different to the classifier despite conveying the same sentiment. Each word occupies its own orthogonal dimension in a [vocab_size]-dimensional space.

With **word embeddings**, "great" and "excellent" have similar vectors because they appear in similar contexts during training. The averaging operation produces nearly identical document vectors for both sentences, so the classifier correctly identifies both as positive. The embedding space is continuous and low-dimensional (e.g., 50–300 dimensions), and similarity is measured by cosine distance.

**When to use each:**

- **Use BOW / TF-IDF when:** You need interpretability (each feature = a specific word), you have limited training data, or you need a fast, cheap baseline. BOW also handles rare words well because they get their own dimension.

- **Use embeddings when:** You have enough data to learn good vectors, you need to handle synonyms and semantic similarity, or you want a dense representation that generalizes across word choices. Embeddings also handle out-of-vocabulary words better if you use subword information (FastText) or pre-trained vectors.

**The key insight:** Embeddings capture *type-level* similarity (all words exist in a continuous space), while BOW captures *token-level* exact match (each word is its own binary/weighted feature). Modern NLP pipelines often use both — TF-IDF features concatenated with embedding-based features.

In this notebook, the embedding model's **most_similar** method shows which words the model considers related — a capability BOW fundamentally lacks.

## Key Takeaways — Placement Interview Revision

1. **Dense vs sparse:** Word embeddings are dense, low-dimensional vectors shaped like [vocab_size x embed_dim]; bag-of-words is sparse, high-dimensional, shaped like [n_docs x vocab_size].

2. **Semantic similarity:** Embeddings capture similarity through vector dot products (cosine similarity); "good" and "excellent" are close in embedding space but orthogonal in BOW space.

3. **Bag-of-embeddings = mean pooling:** Averaging all word vectors in a document is the simplest neural text classifier; it ignores word order but captures semantic content surprisingly well.

4. **Hyperparameter trade-offs:** Embedding dimension controls capacity (too small = underfit, too large = overfit); batch size controls gradient noise vs stability. Always cross-validate.

5. **Embeddings are learned, BOW is counted:** Embedding vectors are optimized via gradient descent on a downstream task; BOW features are based on raw frequency statistics. Learning beats counting when data is abundant.